In [ ]:
# Cell 2 — Install Dependencies

!pip install -q -U \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    sentence-transformers \
    transformers \
    accelerate \
    bitsandbytes \
    faiss-cpu \
    pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [ ]:
from google.colab import userdata
import os

os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Rag Fitter"
os.environ["LANGSMITH_ENDPOINT"] = "https://apac.api.smith.langchain.com"

In [ ]:
# Cell 3 — Verify GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("Enable T4 GPU in Runtime > Change runtime type.")


CUDA available: True
GPU: Tesla T4
VRAM (GB): 14.56


In [ ]:
#  Imports

from langsmith import traceable
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda
)
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

/tmp/ipykernel_3500/76287577.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
# Google Drive

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Project Path

PROJECT_PATH = "/content/drive/MyDrive/RAG_Project"

DATA_PATH = f"{PROJECT_PATH}/data"
VECTOR_DB_PATH = f"{PROJECT_PATH}/vectorstore"

print(DATA_PATH)
print(VECTOR_DB_PATH)

/content/drive/MyDrive/RAG_Project/data
/content/drive/MyDrive/RAG_Project/vectorstore


In [ ]:
# Function: Create RAG Project Directories

import os

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(VECTOR_DB_PATH, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [ ]:
# Cell 5 — Project Path

PDF_PATH = "/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-Alternative-Fuel-For-Spark-Ignition-Engine.pdf"

print(PDF_PATH)

/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-Alternative-Fuel-For-Spark-Ignition-Engine.pdf


In [ ]:
# Trace PDF Loading
@traceable(name="01_load_pdf",run_type="chain")
def load_pdf(path: str):
  return PyPDFLoader(path).load()
docs = load_pdf(PDF_PATH)
print("Pages:",len(docs))
print(docs[0].page_content[:400] if docs else "Laa Laa lalal Laa")

Pages: 8
INTERNATIONAL JOURNAL OF SCIENTIFIC & TECHNOLOGY RESEARCH VOLUME 4, ISSUE 09, SEPTEMBER 2015    ISSN 2277-8616 
179 
IJSTR©2015 
www.ijstr.org 
Development And Testing Of Biogas-Petrol Blend 
As An Alternative Fuel For Spark Ignition Engine 
 
Awogbemi, Omojola, Adeyemo, Sunday Babatunde 
 
Abstract: This research is on the development and testing of a biogas -petrol blend to run a spark ignition 


In [ ]:
# Trace Text Splitting
@traceable(name="02_split_documents",run_type="chain")
def split_documents(documents,chunk_size=500,chunk_overlap=50):
  splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                            chunk_overlap=chunk_overlap)
  return splitter.split_documents(documents)

splits = split_documents(docs)
print("Chunks:",len(splits))


Chunks: 63


In [ ]:
 # Trace embedding + FAISS index construction
# FAISS.from_documents invokes embedding internally; this span groups that work.
@traceable(name="03_embed_and_build_faiss",run_type="chain")
def build_vectorstore(chunks):
  embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-MiniLM-L6-v2",
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 64  # Good practice for GPU to keep it fed
    }
)
  return FAISS.from_documents(chunks,embeddings)

vectorstore = build_vectorstore(splits)
print("FAISS index created")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index created


In [ ]:
# Parent setup + full RAG run (hierarchical tracing)
# Run this cell after load_pdf, split_documents, build_vectorstore,
# format_docs, prompt, and llm have been defined.

@traceable(name="setup_pipeline",tags=["setup"],run_type="chain")
def setup_pipeline(pdf_path:str,chunk_size=500,chunk_overlap=55):
  docs = load_pdf(pdf_path)
  splits = split_documents(
      docs,
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )
  vectorstore = build_vectorstore(splits)
  return vectorstore


In [ ]:
# Hugging Face LLM

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
hf_token = os.getenv("HF_TOKEN")

# Tokenizer input text ko model ke samajhne layak token IDs mein convert karta hai.
# "What is RAG?" ↓ Tokenizer [1032, 374, 451, 30]  # illustrative IDs
tokenizer  = AutoTokenizer.from_pretrained(MODEL_ID,token=hf_token or None)

# AutoTokenizer — model-compatible tokenizer automatically select karta hai.
# from_pretrained() — Hugging Face se tokenizer configuration/files load karta hai; cached files reuse ho sakti hain.

model = AutoModelForCausalLM.from_pretrained(MODEL_ID,
                                             torch_dtype=torch.float16,
                                             device_map="auto",
                                             token = hf_token or None
                                             )

generation_pipeline = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    max_new_tokens = 384,
    do_sample =False,
    return_full_text = False,
    pad_token_id = tokenizer.eos_token_id
)

llm= HuggingFacePipeline(pipeline=generation_pipeline)
print("LLM loaded without Quantization")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM loaded without Quantization


In [ ]:
# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only from the supplied context. Prefer Hinglish. If the context does not contain the answer, say you do not know from this document. Do not invent facts."),
    ("human", "Question: {question}\n\nContext:\n{context}")
])
print("Prompt ready.")


Prompt ready.


In [ ]:
#  Format Retrieved Documents
@traceable(name="05_format_context",run_type="chain")
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
# Trace a standalone retrieval test
@traceable(name="04_retrieve_documents", run_type="chain")
def retriver_documents(question:str):

  return retriever.invoke(question)

In [ ]:
test_question = "What are the main Result of Engine Performance Tests?"
retrieved_docs = retriver_documents(test_question)
print("Retrieved chunks:", len(retrieved_docs))
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Chunk {i} | {doc.metadata} ---\n{doc.page_content[:300]}")

Retrieved chunks: 4

--- Chunk 1 | {'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2015-09-15T23:49:15+05:30', 'author': 'NATA', 'moddate': '2015-09-15T23:49:15+05:30', 'source': '/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-Alternative-Fuel-For-Spark-Ignition-Engine.pdf', 'total_pages': 8, 'page': 1, 'page_label': '2'} ---
AC dynamometer. The technical specifications of the 
engine test rig and the dynamometer are pr esented in 
Tables 1 and 2  respectively. The engine performance 
characteristics were monitored within the speed range of 
1000rpm and 3500rpm varied incrementally by 500rpm. 
The engine was run on petro

--- Chunk 2 | {'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2015-09-15T23:49:15+05:30', 'author': 'NATA', 'moddate': '2015-09-15T23:49:15+05:30', 'source': '/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-

In [ ]:
@traceable(name="06_pdf_rag_full_run",run_type="chain")
def setup_pipeline_and_query(question:str,pdf_path:str):

  vectorstore = setup_pipeline(pdf_path,
        chunk_size=500,
        chunk_overlap=50)


  config = {
      "run_name":"rag_question",
      "tags":["rag","huggingface","faiss"],
      "metadata":{
          "embedding_model":"paraphrase-MiniLM-L6-v2",
          "llm_model": MODEL_ID,
          "retriver_top_k": 4
      }
  }
  retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    )

  parallel = RunnableParallel({
      "context": retriever | RunnableLambda(format_docs),
      "question": RunnablePassthrough(),
  })

  chain = parallel | prompt | llm | StrOutputParser()

  lc_config = {
      "run_name": "pdf_rag_query",
      "tags": ["rag", "question-answering"],
      "metadata": {
          "pdf_name": os.path.basename(pdf_path),
          "chunk_size": 500,
          "chunk_overlap": 50,
          "retriever_top_k": 4
      }
  }
  return chain.invoke(question, lc_config=config)



In [ ]:
# Run full setup + RAG question under one root trace
question = "key things of ABSTRACT discussed in the paper."
answer = setup_pipeline_and_query(question,PDF_PATH)
print(answer)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 
combustion engines [Eastop and McConkey 1969]. 
The utilization of biogas as an alternative fuel has 
been investigated by many researchers [Awogbemi and 
Adeyemo 2013]. Biogas is a mixture of methane and 
carbon dioxide produced by anaerobic digestion of 
organic matter. It is a renewable source of energy that 
can be used as a substitute for natural gas or petrol. 
Biogas can be produced from various organic 
materials such as manure, sewage sludge, food waste, 
and agricultural residues [Eze and Elijah 2014]. 
The objective of this study was to investigate the 
influence of some parameters on biogas yield. The 
parameters investigated were: temperature, 
substrate loading, retention time, and agitation rate. 
The results showed that the highest biogas yield was 
obtained at a temperature of 35°C, substrate loading of 
10%, retention time of 15 days, and agitation rate of 
100 rpm. The study also found that the biogas yield 
decreased with increasing temperature and retention 
time